### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [4]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableMap

In [5]:
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {"device" : "cpu"}

embeddings = HuggingFaceEmbeddings(
    model_name = model_name, model_kwargs = {**model_kwargs,"local_files_only":True }
)


In [6]:
load_dotenv()

llm = ChatGroq(model_name= "openai/gpt-oss-20b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002B1DE6A5DC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002B1B66F89B0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks,embeddings)
retriever = vectorstore.as_retriever(
    search_type ="mmr",search_kwargs = {"k":4,"lambda_mult":0.7}
)

In [14]:
#query decomposition
decomposition_prompt = PromptTemplate.from_template(
     """
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
"""
)
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [15]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question = decomposition_chain.invoke({"question": query})

In [16]:
print(decomposition_question)

**Sub‑questions for document retrieval**

1. **Memory in LangChain**  
   - What types of memory (e.g., vector stores, conversation state, context windows) are available in LangChain, and how are they implemented and configured?

2. **Memory in CrewAI**  
   - What memory mechanisms does CrewAI provide, and how does it manage and persist context across tasks or agents?

3. **Agents in LangChain**  
   - How does LangChain define and orchestrate agents (e.g., tool‑using agents, chain‑of‑thought agents), and what architectural patterns or APIs does it expose for agent development?

4. **Agents in CrewAI**  
   - How does CrewAI implement agent behavior and coordination, and what differences exist in its agent architecture compared to LangChain’s approach?


In [17]:
#step:3 QA chain per question
qa_prompt = PromptTemplate.from_template(
    """
Use the context below to answer the question.

Context:
{context}

Question: {input}
"""
)
qa_chain = create_stuff_documents_chain(llm = llm, prompt = qa_prompt)

In [18]:
# Step 4: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [
        q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()
    ]

    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")

    return "\n\n".join(results)

In [19]:
# Step 5: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: **Decomposed Sub‑Questions**
A: **Decomposed Sub‑Questions**

1. **Composition & Reuse**
   - How does the framework make it easy to compose and reuse chains?
   - What are the “standard patterns” (Stuff, Map‑Reduce, Refine) and how do they differ?

2. **Traceability & Debugging**
   - In what ways does the framework support full traceability of agent decisions and interactions?
   - How does traceability improve debugging and transparency compared to standalone agent setups?
   - What specific tools or logs does the framework provide to trace agent actions?

3. **Scaling & Collaboration**
   - What does “structured agent collaboration” mean in the context of CrewAI?
   - How does CrewAI enable horizontal scaling (adding more agents)?
   - How does CrewAI enable vertical scaling (deeper reasoning within a single agent or team)?

4. **Version Context**
   - What changes were introduced between v1 and v9 regarding traceability or other features?
   - Are there any bac